> Bản dịch tiếng Việt · nguồn: [RLHF.ipynb](../../documents/chapter11/RLHF.ipynb) · 《动手学大模型》(Lordog/dive-into-llms)

# Thực hành RLHF
> Sổ tay thực nghiệm này được dịch và tổng hợp từ các tài liệu trên mạng: [blog](https://newfacade.github.io/notes-on-reinforcement-learning/17-ppo-trl.html) & [trl examples](https://github.com/huggingface/trl/blob/main/examples/notebooks/gpt2-sentiment.ipynb)

Cấu hình tái hiện thí nghiệm: một card NVDIA A800-SXM4-80GB chiếm 10097MiB, thời gian huấn luyện 35min19s.

## PPO hoạt động như thế nào
1. Rollout: mô hình ngôn ngữ sinh phản hồi dựa trên query.
2. Evaluation: truy vấn và phản hồi được đánh giá bằng hàm, mô hình, phản hồi của con người hoặc một tổ hợp nào đó của chúng. Quá trình này nên sinh ra một **giá trị vô hướng** cho mỗi cặp truy vấn/phản hồi.
3. Optimization: trong bước tối ưu, các cặp truy vấn/phản hồi được dùng để tính log-probability của các token trong chuỗi. Việc này được thực hiện thông qua mô hình được huấn luyện và mô hình tham chiếu. Độ phân kỳ KL giữa hai đầu ra được dùng làm tín hiệu thưởng bổ sung, nhằm bảo đảm các phản hồi được sinh ra không lệch quá xa khỏi mô hình ngôn ngữ tham chiếu. Sau đó, PPO được dùng để huấn luyện mô hình ngôn ngữ policy.
<div style="text-align: center">
<img src='../../documents/chapter11/figs/trl1.png' width='600'>
<p style="text-align: center;"> <b>Hình:</b> Sơ đồ quy trình PPO </p>
</div>

# Tinh chỉnh GPT-2 để sinh bình luận tích cực  
> Tối ưu GPT-2 để sinh các bình luận phim IMDB tích cực, bằng cách sử dụng bộ phân loại cảm xúc BERT làm hàm thưởng.

<div style="text-align: center">
<img src='../../documents/chapter11/figs/gpt2_bert_training.png' width='600'>
<p style="text-align: center;"> <b>Hình:</b> Thiết lập thực nghiệm tinh chỉnh GPT-2</p>
</div>

Chúng tôi tinh chỉnh GPT-2 để sinh các bình luận phim tích cực dựa trên tập dữ liệu IMDB. Mô hình sẽ nhận phần mở đầu của một bình luận có thật và cần sinh ra phần tiếp theo mang tính tích cực. Để thưởng cho phần tiếp theo tích cực, chúng tôi sử dụng bộ phân loại BERT để phân tích cảm xúc của câu được sinh ra, và lấy đầu ra của bộ phân loại làm tín hiệu thưởng cho quá trình huấn luyện PPO.

## Thiết lập thực nghiệm

### Tải mô hình và dữ liệu
Tập dữ liệu
```bash
export HF_ENDPOINT=https://hf-mirror.com; huggingface-cli download --resume-download stanfordnlp/imdb --local-dir dataset/imdb --repo-type dataset
```
Mô hình tham chiếu
```bash
export HF_ENDPOINT=https://hf-mirror.com; huggingface-cli download --resume-download lvwerra/gpt2-imdb --local-dir model/gpt2-imdb
```
Mô hình thưởng
```bash
export HF_ENDPOINT=https://hf-mirror.com; huggingface-cli download --resume-download lvwerra/distilbert-imdb --local-dir model/distilbert-imdb
```

### Nhập các thư viện phụ thuộc

In [ ]:
# %pip install -r requirements.txt
# import os
# os.environ['CUDA_VISIBLE_DEVICES'] = '7'

In [ ]:
import torch
from tqdm import tqdm
import pandas as pd

tqdm.pandas()

from transformers import pipeline, AutoTokenizer
from datasets import load_dataset

from trl import PPOTrainer, PPOConfig, AutoModelForCausalLMWithValueHead
from trl.core import LengthSampler

### Cấu hình

In [ ]:
config = PPOConfig(
    model_name="model/gpt2-imdb",
    learning_rate=1.41e-5,
    log_with="wandb",
)

sent_kwargs = {"top_k": None, "function_to_apply": "none", "batch_size": 16}

In [ ]:
import wandb

wandb.init()

Bạn có thể thấy chúng tôi đã nạp một mô hình GPT-2 có tên `gpt2_imdb`. Mô hình này được tinh chỉnh thêm 1 epoch trên tập dữ liệu IMDB bằng [script](https://github.com/huggingface/transformers/blob/main/examples/legacy/run_language_modeling.py) của Hugging Face (không có thiết lập đặc biệt). Các tham số còn lại được lấy chủ yếu từ bài báo gốc "[Fine-Tuning Language Models from Human Preferences](https://huggingface.co/papers/1909.08593)". Mô hình này cũng như mô hình BERT đều có sẵn trong kho mô hình của Hugging Face, đường link cụ thể ở [đây](https://huggingface.co/models).

## Nạp dữ liệu và mô hình

### Nạp tập dữ liệu IMDB  
Tập dữ liệu IMDB bao gồm 50,000 bình luận phim được gán nhãn cảm xúc "tích cực"/"tiêu cực". Chúng tôi nạp tập dữ liệu IMDB vào một DataFrame và lọc lấy các bình luận có ít nhất 200 ký tự. Sau đó, chúng tôi tách token từng văn bản và dùng `LengthSampler` để cắt ngẫu nhiên về độ dài được chỉ định.

In [ ]:
def build_dataset(
    config,
    dataset_name="dataset/imdb",
    input_min_text_length=2,
    input_max_text_length=8,
):
    """
    Build dataset for training. This builds the dataset from `load_dataset`, one should
    customize this function to train the model on its own dataset.

    Args:
        dataset_name (`str`):
            The name of the dataset to be loaded.

    Returns:
        dataloader (`torch.utils.data.DataLoader`):
            The dataloader for the dataset.
    """
    tokenizer = AutoTokenizer.from_pretrained(config.model_name)
    tokenizer.pad_token = tokenizer.eos_token
    # load imdb with datasets
    ds = load_dataset(dataset_name, split="train")
    ds = ds.rename_columns({"text": "review"})
    ds = ds.filter(lambda x: len(x["review"]) > 200, batched=False)

    input_size = LengthSampler(input_min_text_length, input_max_text_length)

    def tokenize(sample):
        sample["input_ids"] = tokenizer.encode(sample["review"])[: input_size()]
        sample["query"] = tokenizer.decode(sample["input_ids"])
        return sample

    ds = ds.map(tokenize, batched=False)
    ds.set_format(type="torch")
    return ds

In [ ]:
dataset = build_dataset(config)


def collator(data):
    return dict((key, [d[key] for d in data]) for key in data[0])

### Nạp mô hình ngôn ngữ GPT2 đã pretrain
Chúng tôi nạp mô hình GPT2 kèm đầu giá trị (value head) và bộ tách token. Chúng tôi nạp mô hình hai lần; mô hình thứ nhất dùng để tối ưu, còn mô hình thứ hai đóng vai trò tham chiếu, dùng để tính độ phân kỳ KL (KL-divergence) với điểm khởi tạo. Điều này đóng vai trò như một tín hiệu thưởng bổ sung trong huấn luyện PPO, nhằm bảo đảm mô hình sau tối ưu không lệch quá xa khỏi mô hình ngôn ngữ ban đầu.

In [ ]:
model = AutoModelForCausalLMWithValueHead.from_pretrained(config.model_name)
ref_model = AutoModelForCausalLMWithValueHead.from_pretrained(config.model_name)
tokenizer = AutoTokenizer.from_pretrained(config.model_name)

tokenizer.pad_token = tokenizer.eos_token

### Khởi tạo PPOTrainer  
`PPOTrainer` chịu trách nhiệm phân bổ thiết bị và tối ưu cho các bước tiếp theo:

In [ ]:
ppo_trainer = PPOTrainer(
    config, model, ref_model, tokenizer, dataset=dataset, data_collator=collator
)

### Nạp bộ phân loại BERT  
Chúng tôi nạp một bộ phân loại BERT đã được tinh chỉnh trên tập dữ liệu IMDB.

In [ ]:
device = ppo_trainer.accelerator.device
if ppo_trainer.accelerator.num_processes == 1:
    device = 0 if torch.cuda.is_available() else "cpu"  # to avoid a `pipeline` bug
sentiment_pipe = pipeline(
    "sentiment-analysis", model="model/distilbert-imdb", device=device
)

Đầu ra của mô hình là logits của lớp tiêu cực và lớp tích cực. Chúng tôi sẽ dùng logits của lớp tích cực làm tín hiệu thưởng cho mô hình ngôn ngữ.

In [ ]:
text = "this movie was really bad!!"
sentiment_pipe(text, **sent_kwargs)

In [ ]:
text = "this movie was really good!!"
sentiment_pipe(text, **sent_kwargs)

### Thiết lập sinh văn bản  
Đối với việc sinh phản hồi, chúng tôi chỉ sử dụng phương pháp sampling, đồng thời bảo đảm tắt top-k và nucleus sampling, kèm với việc đặt một độ dài tối thiểu.

In [ ]:
gen_kwargs = {
    "min_length": -1,
    "top_k": 0.0,
    "top_p": 1.0,
    "do_sample": True,
    "pad_token_id": tokenizer.eos_token_id,
}

## Tối ưu mô hình

### Vòng lặp huấn luyện

Vòng lặp huấn luyện bao gồm các bước chính sau:
1. Lấy phản hồi theo truy vấn từ mạng policy (GPT-2)  
2. Lấy cảm xúc của truy vấn/phản hồi từ BERT  
3. Tối ưu policy bằng PPO, sử dụng bộ ba (truy vấn, phản hồi, thưởng)  
<!-- **Thời gian huấn luyện**  
Trong thiết lập đã chỉ định ở trên, bước này cần **khoảng 2 giờ** trên GPU V100. -->

In [ ]:
output_min_length = 4
output_max_length = 16
output_length_sampler = LengthSampler(output_min_length, output_max_length)


generation_kwargs = {
    "min_length": -1,
    "top_k": 0.0,
    "top_p": 1.0,
    "do_sample": True,
    "pad_token_id": tokenizer.eos_token_id,
}


for epoch, batch in enumerate(tqdm(ppo_trainer.dataloader)):
    query_tensors = batch["input_ids"]

    #### Get response from gpt2
    response_tensors = []
    for query in query_tensors:
        gen_len = output_length_sampler()
        generation_kwargs["max_new_tokens"] = gen_len
        query_response = ppo_trainer.generate(query, **generation_kwargs).squeeze()
        response_len = len(query_response) - len(query)
        response_tensors.append(query_response[-response_len:])
    batch["response"] = [tokenizer.decode(r.squeeze()) for r in response_tensors]

    #### Compute sentiment score
    texts = [q + r for q, r in zip(batch["query"], batch["response"])]
    pipe_outputs = sentiment_pipe(texts, **sent_kwargs)
    positive_scores = [
        item["score"]
        for output in pipe_outputs
        for item in output
        if item["label"] == "POSITIVE"
    ]
    rewards = [torch.tensor(score) for score in positive_scores]

    #### Run PPO step
    stats = ppo_trainer.step(query_tensors, response_tensors, rewards)
    ppo_trainer.log_stats(stats, batch, rewards)

### Tiến độ huấn luyện  
Nếu bạn đang dùng Weights & Biases để theo dõi tiến độ huấn luyện, bạn sẽ thấy các đường cong tương tự như hình dưới đây. Xem báo cáo mẫu có tính tương tác trên wandb.ai: [liên kết](https://wandb.ai/huggingface/trl/runs/w9l3110g).  
<div style="text-align: center">
<img src='../../documents/chapter11/figs/gpt2_tuning_progress.png' width='800'>
<p style="text-align: center;"> <b>Hình:</b> Sự tiến hóa của mức thưởng trung bình trong quá trình huấn luyện </p>
</div>  
Có thể quan sát thấy rằng sau vài bước tối ưu, mô hình bắt đầu sinh ra đầu ra tích cực hơn.  

<!-- > Lưu ý: nếu nghiên cứu độ phân kỳ KL, bạn có thể thấy mô hình lúc này vẫn chưa hội tụ tới độ phân kỳ KL mục tiêu. Để đạt được mục tiêu cần thời gian huấn luyện dài hơn hoặc hệ số khởi tạo cao hơn. -->

## Kiểm tra mô hình  
Hãy kiểm tra một vài ví dụ từ tập dữ liệu IMDB. Chúng tôi có thể dùng `ref_model` để so sánh mô hình sau tối ưu `model` với mô hình trước khi tối ưu.

In [ ]:
#### get a batch from the dataset
bs = 16
game_data = dict()
dataset.set_format("pandas")
df_batch = dataset[:].sample(bs)
game_data["query"] = df_batch["query"].tolist()
query_tensors = df_batch["input_ids"].tolist()

response_tensors_ref, response_tensors = [], []

#### get response from gpt2 and gpt2_ref
for i in range(bs):
    query = torch.tensor(query_tensors[i]).to(device)

    gen_len = output_length_sampler()
    query_response = ref_model.generate(
        query.unsqueeze(0), max_new_tokens=gen_len, **gen_kwargs
    ).squeeze()
    response_len = len(query_response) - len(query)
    response_tensors_ref.append(query_response[-response_len:])

    query_response = model.generate(
        query.unsqueeze(0), max_new_tokens=gen_len, **gen_kwargs
    ).squeeze()
    response_len = len(query_response) - len(query)
    response_tensors.append(query_response[-response_len:])

#### decode responses
game_data["response (before)"] = [
    tokenizer.decode(response_tensors_ref[i]) for i in range(bs)
]
game_data["response (after)"] = [
    tokenizer.decode(response_tensors[i]) for i in range(bs)
]

#### sentiment analysis of query/response pairs before/after
texts = [q + r for q, r in zip(game_data["query"], game_data["response (before)"])]
pipe_outputs = sentiment_pipe(texts, **sent_kwargs)
positive_scores = [
    item["score"]
    for output in pipe_outputs
    for item in output
    if item["label"] == "POSITIVE"
]
game_data["rewards (before)"] = positive_scores

texts = [q + r for q, r in zip(game_data["query"], game_data["response (after)"])]
pipe_outputs = sentiment_pipe(texts, **sent_kwargs)
positive_scores = [
    item["score"]
    for output in pipe_outputs
    for item in output
    if item["label"] == "POSITIVE"
]
game_data["rewards (after)"] = positive_scores

# store results in a dataframe
df_results = pd.DataFrame(game_data)
df_results

Qua việc quan sát giá trị trung bình/trung vị của thưởng trên các chuỗi được sinh ra, chúng tôi nhận thấy sự khác biệt rõ rệt.

In [ ]:
print("mean:")
display(df_results[["rewards (before)", "rewards (after)"]].mean())
print()
print("median:")
display(df_results[["rewards (before)", "rewards (after)"]].median())

## Lưu mô hình  
Cuối cùng, chúng tôi lưu mô hình để sử dụng về sau.

In [ ]:
model.save_pretrained("model/gpt2-imdb-pos-v2")
tokenizer.save_pretrained("model/gpt2-imdb-pos-v2")